# Lecture 1.7 — Reading a RunResult: `final_output`, `last_agent`, `usage`

**OpenAI Agents SDK — Complete Course | Section 01: Introduction & Setup**

---

In the previous lecture (1.6), we ran our first agent and got a glimpse of the `RunResult` object — printing `final_output` and `last_agent.name`. In this notebook, we slow down and **properly unpack the `RunResult` in full**.

Understanding what comes back from a run is fundamental to everything in Sections 2 through 6. By the end of this notebook you will know:

- What `final_output` is, its type, and why it changes in Lecture 2.5
- What `last_agent` tells you and why it matters in multi-agent systems
- How to read token usage via `result.context_wrapper.usage`
- What `new_items` contains and how it connects to streaming (Section 4)
- How `to_input_list()` lets you chain runs together (the foundation of Section 5)

> **Note for local users:** If you are running this notebook locally rather than in Google Colab, set your API key as an environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY="your-key-here"
> ```

## Cell 1 — Install the SDK

This cell installs the `openai-agents` package, pinned to a specific version. If that exact version is already present in this environment, pip confirms it and moves on instantly. If not, pip installs it, so this notebook behaves exactly as it was built and recorded.

**Always run this cell first**, even if you believe the package is already installed — a Colab runtime restart wipes every installed package, even though old cell outputs stay visible on screen.

In [ ]:
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.8/850.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.9 MB/s eta 0:00:00


## Cell 2 — Imports and API Key Setup

We import two things from the SDK:

| Import | Purpose |
|--------|---------|
| `Agent` | Creates an agent with a name, instructions, and optional tools |
| `Runner` | Executes the agent — we use `await Runner.run()` in notebooks |

**Setting the API key in Google Colab:**
1. Click the **🔑 Secrets** icon in the left sidebar (or go to *Tools → Secrets*).
2. Click **+ Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your OpenAI key as the value.
4. Toggle **Notebook access** to ON.
5. Run this cell — `userdata.get("OPENAI_API_KEY")` retrieves it and writes it to `os.environ` so the SDK can pick it up automatically.

> **Local users:** Skip the `userdata` lines. The SDK reads `os.environ["OPENAI_API_KEY"]` automatically if you exported it in your terminal.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from agents import Agent, Runner

## Cell 3 — Define the Agent

We create a simple summarisation agent. The focus of this lecture is entirely on the **result object** — not on what the agent says. That is why we use a straightforward task (summarising a concept in 2–3 sentences) that produces a predictable, single-turn output with no tool calls.

| Parameter | Value | Why |
|-----------|-------|-----|
| `name` | `"Summariser"` | Identifies this agent in `last_agent.name` and traces |
| `instructions` | Summarise in 2–3 sentences | Keeps the output short and the run simple |
| `output_type` | *(not set)* | Defaults to `str` — `final_output` will be a plain string |

Notice that we are not setting `output_type`. In Lecture 2.5 we will set it to a Pydantic model, and that is when `final_output` stops being a string and becomes a structured object.

In [ ]:
agent = Agent(
    name="Summariser",
    instructions=(
        "You are a helpful assistant that summarises "
        "text concisely in 2-3 sentences."
    ),
)

## Cell 4 — Run the Agent and Capture the Result

We use `await Runner.run()` — the correct method for Jupyter notebooks. Colab (and all Jupyter environments) run an event loop in the background, which means `Runner.run_sync()` would raise a `RuntimeError`. Using `await Runner.run()` works naturally with the existing loop.

| Method | When to use |
|--------|-------------|
| `await Runner.run(agent, input)` | **Notebooks and async contexts** (FastAPI, async frameworks) |
| `Runner.run_sync(agent, input)` | Plain Python scripts only — raises an error if an event loop is already running |
| `Runner.run_streamed(agent, input)` | When you need token-by-token streaming (Section 4) |

The **entire return value** is stored in `result`. Everything from this point onwards is about inspecting that object — we are not printing anything yet.

In [ ]:
result = await Runner.run(
    agent,
    "Explain what a large language model is.",
)

## Cell 5 — Explore `final_output`

`final_output` is the property you will use most often. It holds the final answer produced by the last agent that ran.

**What type is it?** That depends on whether you set `output_type` on the agent:

| `output_type` set? | Type of `final_output` |
|--------------------|------------------------|
| No (default) | `str` |
| Yes (e.g. a Pydantic model) | Instance of that model |
| Run stopped early (e.g. approval interruption) | `None` |

Right now we have not set `output_type`, so `final_output` is always a `str`. In Lecture 2.5 we will set it to a Pydantic model — and that is when structured outputs become really powerful.

In [ ]:
print(type(result.final_output))
print(result.final_output)

<class 'str'>
A large language model is an AI system trained on huge amounts of text to learn patterns in language. It can predict and generate text, answer questions, summarize content, and help with many writing and reasoning tasks.


## Cell 6 — Explore `last_agent`

`last_agent` tells you **which agent produced the final output**. In a single-agent run like this one, it is always the same agent we started with. So why does the SDK expose this?

In Section 5 we will build systems where agents hand off to other agents. The agent that *started* a run is not necessarily the one that *finished* it. `last_agent` is what you use to determine which agent should handle the **next user turn** — and to debug which agent produced an unexpected output.

For now, notice that `last_agent` is a full `Agent` object — it has `.name`, `.instructions`, `.model`, and every other property. We are printing just `.name` here, but in a real system you might inspect `.instructions` or `.tools`.

In [ ]:
print(result.last_agent.name)
print(type(result.last_agent))

Summariser
<class 'agents.agent.Agent'>


## Cell 7 — Explore Usage via `context_wrapper`

Token usage is accessed via **`result.context_wrapper.usage`** — not `result.usage` directly. The `context_wrapper` is the runtime state container for the entire run. Usage lives inside it because usage is tracked across all turns and all agents in a single run.

The `Usage` object has the following properties:

| Property | What it tells you |
|----------|-------------------|
| `requests` | Total number of LLM API calls made during this run |
| `input_tokens` | Total input tokens sent across all requests |
| `output_tokens` | Total output tokens received across all requests |
| `total_tokens` | `input_tokens + output_tokens` |

**Why this matters:** Token costs compound fast in multi-agent systems. Get into the habit of tracking `requests` and `total_tokens`. When your system runs 20 turns across 4 agents, every one of those model calls adds to this count.

In [ ]:
usage = result.context_wrapper.usage

print("Requests:      ", usage.requests)
print("Input tokens:  ", usage.input_tokens)
print("Output tokens: ", usage.output_tokens)
print("Total tokens:  ", usage.total_tokens)

Requests:       1
Input tokens:   37
Output tokens:  46
Total tokens:   83


## Cell 8 — Per-Request Usage Breakdown

The `usage` object also contains a list called `request_usage_entries`. Each entry holds the token counts for a **single LLM API call**. This is useful for:

- **Cost calculation:** Pricing varies by model and by prompt vs. completion tokens. Having the per-call breakdown lets you compute exact costs.
- **Debugging:** If one turn is unexpectedly expensive, you can identify which specific model call caused it.
- **Context window management:** You can see exactly how much context was consumed per turn.

In a simple single-turn run like this one, the list has **one entry**. In a multi-turn or multi-agent run, the list grows — one entry per model call, across all agents and all turns.

In [ ]:
for i, req in enumerate(result.context_wrapper.usage.request_usage_entries):
    print(
        f"Request {i + 1}: "
        f"{req.input_tokens} in, "
        f"{req.output_tokens} out"
    )

Request 1: 37 in, 46 out


## Cell 9 — Explore `new_items`

`new_items` is the **full transcript** of everything that happened during this run — model responses, tool calls, tool outputs, handoffs. It is a list of `RunItem` objects.

In a simple single-turn run with no tools, this list is short — typically just one `MessageOutputItem` representing the model's response. As your agents grow more complex, `new_items` grows with them:

| Scenario | What `new_items` contains |
|----------|---------------------------|
| Simple single-turn | 1 `MessageOutputItem` |
| Agent uses a tool | `ToolCallItem` + `ToolOutputItem` + `MessageOutputItem` |
| Agent hands off | `HandoffCallItem` + items from the next agent |

In Section 4 (Streaming), `new_items` is the basis for the streaming event system — each item in this list corresponds to one or more streaming events.

In [ ]:
print(f"Number of new items: {len(result.new_items)}")
for item in result.new_items:
    print(item)
    print(type(item).__name__)

Number of new items: 1
MessageOutputItem(agent=Agent(name='Summariser', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a helpful assistant that summarises text concisely in 2-3 sentences.', prompt=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_022d305111503dec006a375c45477c8192abddfae504689da7', con

## Cell 10 — Explore `to_input_list()`

`to_input_list()` converts the result of this run into a list of input items that can be passed **directly into the next `Runner.run()` call**. This is how you chain one run into the next — the foundation of multi-turn conversations.

The returned list contains:
- The original user message
- All items generated during this run (model responses, tool calls, etc.)

You will use this pattern constantly in Section 5:

```python
# Turn 1
result1 = await Runner.run(agent, "Hello")

# Turn 2 — pass the full history forward
result2 = await Runner.run(agent, result1.to_input_list() + [{"role": "user", "content": "Follow up"}])
```

For now, we just print the list to see its structure.

In [ ]:
input_list = result.to_input_list()

print(f"Input list length: {len(input_list)}")
print(input_list)

Input list length: 2
[{'content': 'Explain what a large language model is.', 'role': 'user'}, {'id': 'msg_022d305111503dec006a375c45477c8192abddfae504689da7', 'content': [{'annotations': [], 'text': 'A large language model is an AI system trained on huge amounts of text to learn patterns in language. It can predict and generate text, answer questions, summarize content, and help with many writing and reasoning tasks.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}]


## Quick Reference — `RunResult` Surfaces

Here is a summary of all the key `RunResult` properties and methods covered in this lecture.

| Property / Method | What it gives you |
|-------------------|-------------------|
| `final_output` | The agent's final answer (`str` by default, or `last_agent.output_type` if set, or `None`) |
| `last_agent` | The `Agent` object that produced the final output |
| `context_wrapper.usage` | Aggregated token counts and per-request breakdown for the run |
| `context_wrapper.usage.requests` | Number of LLM API calls made |
| `context_wrapper.usage.input_tokens` | Total input tokens across all requests |
| `context_wrapper.usage.output_tokens` | Total output tokens across all requests |
| `context_wrapper.usage.total_tokens` | `input_tokens + output_tokens` |
| `context_wrapper.usage.request_usage_entries` | Per-request token breakdown list |
| `new_items` | All `RunItem` objects generated during the run (messages, tool calls, handoffs) |
| `input` | The base input used for this run segment |
| `last_response_id` | Response ID of the last model call (for Responses API chaining) |
| `to_input_list()` | Input-item view of the run — used to chain into the next turn |
| `final_output_as(cls)` | Convenience cast of `final_output` to a specific type |

---

**What this notebook did NOT cover (coming in later lectures):**
- Structured outputs / Pydantic `output_type` → Lecture 2.5
- Streaming result events → Section 4
- Sessions and cross-run memory → Update U1
- `RunResultStreaming` → Section 4
- Tool call items in depth → Section 3
- Interruptions and human-in-the-loop → Update U3